# 04 — TGN Training on 3W Dataset

**Goal:** Train and evaluate the Temporal Graph Network on the 3W oil well dataset.

## Key differences from Use Case 1 (synthetic data):
- Real industrial data with genuine noise and missing values
- Multiple wells as graph nodes (generalizable model)
- PRECEDES and CAUSALLY_COUPLED relations used by TGN
- Strict temporal split to prevent data leakage
- Weighted loss for class imbalance (rare events)

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.metrics import classification_report, roc_auc_score
from sklearn.preprocessing import StandardScaler
import sys
sys.path.append('../../src')
from config import DATA_ROOT_3W, PROCESSED_DIR

torch.manual_seed(42)
np.random.seed(42)

PROCESSED_DIR = Path('../../data/processed')
MODELS_DIR    = Path('../../models')
MODELS_DIR.mkdir(exist_ok=True)

print(f'PyTorch: {torch.__version__}')
print(f'Device: {"GPU" if torch.cuda.is_available() else "CPU"}')

## 1. Load Data from Neo4j

In [ ]:
from neo4j import GraphDatabase

NEO4J_URI      = 'bolt://172.22.43.151:7687'
NEO4J_USER     = 'neo4j'
NEO4J_PASSWORD = 'your_password'

def fetch_tgn_data_from_graph() -> pd.DataFrame:
    """
    Fetches observation events from Neo4j in TGN format.
    Each row = one temporal event on an edge (sensor→observation).
    """
    driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))
    with driver.session() as session:
        result = session.run('''
            MATCH (w:Well)-[:HAS_SENSOR]->(s:Sensor)-[:MADE_OBSERVATION]->(o:Observation)
            RETURN w.id AS well_id,
                   s.id AS sensor_id,
                   o.value AS value,
                   o.valid_from AS timestamp,
                   o.is_anomaly AS is_anomaly,
                   o.event_type AS event_type,
                   o.quality_flag AS quality_flag
            ORDER BY o.valid_from
        ''')
        rows = [dict(r) for r in result]
    driver.close()
    df = pd.DataFrame(rows)
    print(f'✅ Fetched {len(df):,} events from Neo4j')
    return df

# Load from Neo4j or from parquet if available
parquet_path = PROCESSED_DIR / 'observations_3w.parquet'
if parquet_path.exists():
    df = pd.read_parquet(parquet_path)
    print(f'✅ Loaded {len(df):,} observations from parquet')
else:
    print('⚠️  Run 02_preprocessing.ipynb and 03_load_neo4j.ipynb first')
    df = pd.DataFrame()

## 2. Prepare TGN Data

In [ ]:
def prepare_tgn_data(df: pd.DataFrame):
    """
    Converts DataFrame to TGN event format.
    
    Split strategy: temporal (no random split to prevent leakage)
    Ref: Longa et al. [7] — future-transductive setting
    """
    if df.empty:
        return None, None, 0

    # Node index mapping
    wells   = df['well_id'].unique().tolist() if 'well_id' in df.columns else []
    sensors = df['sensor_id'].unique().tolist() if 'sensor_id' in df.columns else []
    all_nodes = wells + sensors
    node2idx  = {n: i for i, n in enumerate(all_nodes)}

    # Normalize values
    df = df.copy()
    scaler = StandardScaler()
    if 'value' in df.columns:
        df['value_norm'] = scaler.fit_transform(df[['value']].fillna(0))
    else:
        df['value_norm'] = 0

    # Temporal features
    df['ts_sec'] = pd.to_datetime(df['timestamp'] if 'timestamp' in df.columns else df['valid_from']).astype(np.int64) // 10**9
    df = df.sort_values('ts_sec')
    df['delta_t'] = df.groupby('sensor_id')['ts_sec'].diff().fillna(0)
    df['delta_t_norm'] = df['delta_t'] / (df['delta_t'].max() + 1e-8)

    # Temporal split 70/30
    timestamps = df['ts_sec'].unique()
    split_ts   = timestamps[int(len(timestamps) * 0.7)]
    train_df   = df[df['ts_sec'] <= split_ts]
    test_df    = df[df['ts_sec'] > split_ts]

    def to_tensors(d):
        if 'well_id' in d.columns:
            src = torch.tensor([node2idx.get(w, 0) for w in d['well_id']], dtype=torch.long)
        else:
            src = torch.zeros(len(d), dtype=torch.long)
        dst = torch.tensor([node2idx.get(s, 0) for s in d['sensor_id']], dtype=torch.long)
        ef  = torch.tensor(d[['value_norm','delta_t_norm']].values, dtype=torch.float32)
        dt  = torch.tensor(d['delta_t_norm'].values, dtype=torch.float32).unsqueeze(1)
        y   = torch.tensor(d['is_anomaly'].fillna(False).astype(int).values, dtype=torch.float32)
        return src, dst, ef, dt, y

    print(f'Train: {len(train_df):,} events | Test: {len(test_df):,} events')
    print(f'Anomaly rate train: {train_df["is_anomaly"].mean()*100:.1f}%')
    print(f'Anomaly rate test:  {test_df["is_anomaly"].mean()*100:.1f}%')
    return to_tensors(train_df), to_tensors(test_df), len(all_nodes)

if not df.empty:
    train_data, test_data, num_nodes = prepare_tgn_data(df)
else:
    print('⚠️  Load data first')

## 3. Train TGN

In [ ]:
# Import TGN from src/models
try:
    from models.tgn import TGN, train_tgn, evaluate_tgn
    print('✅ TGN imported from src/models')
except ImportError:
    print('⚠️  TGN not found in src/models — copy TGN-anomaly_detection.py there first')

if not df.empty and train_data is not None:
    model = TGN(
        num_nodes=num_nodes,
        memory_dim=64,    # larger than UC1 — more complex data
        message_dim=64,
        embed_dim=64,
        edge_dim=2,
    )
    print(f'TGN parameters: {sum(p.numel() for p in model.parameters()):,}')

    print('\n🏋️  Training TGN on 3W dataset...')
    train_tgn(model, train_data, epochs=5, batch_size=512)

    print('\n📊 Evaluation:')
    evaluate_tgn(model, test_data)

    # Save checkpoint
    checkpoint_path = MODELS_DIR / 'tgn_3w_checkpoint.pt'
    torch.save({
        'model_state': model.state_dict(),
        'num_nodes': num_nodes,
        'dataset': '3W',
        'epochs': 5,
    }, checkpoint_path)
    print(f'\n✅ Checkpoint saved: {checkpoint_path}')
else:
    print('⚠️  Load data first')